# Prediction ledger check

This notebook verifies the model-response and ledger transformations without calling an LLM, submitting a prediction, or writing to Modal.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from predict import PROMPT_VERSION, Prediction, _normalize_percentile

## 1. Parse a simulated model response

The simulated model deliberately returns `73` to exercise the required 0–100 scale guard.

In [ ]:
model_json = '''
{
  "percentile": 73,
  "confidence": "high",
  "rules_applied": ["Q3-CAL-01", "GLB-GUID-01"]
}
'''

result = Prediction.model_validate_json(model_json)
result.predicted_percentile = _normalize_percentile(
    result.predicted_percentile
)
result

## 2. Construct the internal prediction row

In [ ]:
event = {
    "event_id": "example-event-001",
    "event_type": "EARNINGS_RELEASE",
}
ticker = "AAPL"

detailed_row = {
    "identifier_value": ticker,
    "predicted_percentile": result.predicted_percentile,
    "confidence": result.confidence,
    "rules_applied": result.rules_applied,
    "prompt_version": PROMPT_VERSION,
}
detailed_row

## 3. Construct the competition submission

Only these two fields should leave the prediction worker for each asset.

In [ ]:
submission_row = {
    "identifier_value": detailed_row["identifier_value"],
    "predicted_percentile": detailed_row["predicted_percentile"],
}
submission_row

## 4. Construct the persistent ledger row

Realized values begin as `None` and can be filled by a later outcomes job. The deterministic key prevents webhook retries from creating duplicate records.

In [ ]:
ledger_key = f'{event["event_id"]}:{ticker}'
ledger_row = {
    "event_id": event["event_id"],
    "ticker": detailed_row["identifier_value"],
    "prompt_version": detailed_row["prompt_version"],
    "predicted_percentile": detailed_row["predicted_percentile"],
    "confidence": detailed_row["confidence"],
    "rules_applied": detailed_row["rules_applied"],
    "realized_abnormal": None,
    "realized_percentile": None,
}

print("Ledger key:", ledger_key)
ledger_row

## 5. Run validation assertions

In [ ]:
EXPECTED_LEDGER_COLUMNS = {
    "event_id",
    "ticker",
    "prompt_version",
    "predicted_percentile",
    "confidence",
    "rules_applied",
    "realized_abnormal",
    "realized_percentile",
}
EXPECTED_SUBMISSION_COLUMNS = {
    "identifier_value",
    "predicted_percentile",
}

assert result.predicted_percentile == 0.73
assert set(ledger_row) == EXPECTED_LEDGER_COLUMNS
assert set(submission_row) == EXPECTED_SUBMISSION_COLUMNS
assert 0.0 <= ledger_row["predicted_percentile"] <= 1.0
assert ledger_row["confidence"] == "high"
assert ledger_row["rules_applied"] == [
    "Q3-CAL-01",
    "GLB-GUID-01",
]
assert ledger_row["realized_abnormal"] is None
assert ledger_row["realized_percentile"] is None
assert ledger_key == "example-event-001:AAPL"

print("Ledger validation passed.")